In [1]:
import os
import numpy as np
import pandas as pd
import requests

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
BASE_YEARS = [2021, 2022]   # the two years before delivery; 2024 itself would be look-ahead
TZ = "Europe/Berlin"

In [3]:
def smard_prices(start, end):
    cache = f"{path}/smard_da_hourly.csv"
    if os.path.exists(cache):
        s = pd.read_csv(cache, index_col=0).iloc[:, 0]
        s.index = pd.DatetimeIndex(s.index)
        return s.loc[start:end]

    base, fid, region = "https://www.smard.de/app/chart_data", 4169, "DE-LU"
    weeks = requests.get(f"{base}/{fid}/{region}/index_hour.json").json()["timestamps"]
    rows = []
    for ts in weeks:
        t = pd.Timestamp(ts, unit="ms", tz="UTC")
        if start - pd.Timedelta(days=8) <= t <= end:
            rows += requests.get(f"{base}/{fid}/{region}/{fid}_{region}_hour_{ts}.json").json()["series"]

    a = pd.DataFrame(rows, columns=["ms", "price"]).dropna()
    s = pd.Series(a["price"].to_numpy(dtype=float), index=pd.to_datetime(a["ms"], unit="ms", utc=True))
    s = s[~s.index.duplicated()].sort_index()
    s.to_frame("price").to_csv(cache)
    return s.loc[start:end]


# p = smard_prices(pd.Timestamp(f"{BASE_YEARS[0]}-01-01", tz="UTC"),
#                  pd.Timestamp(f"{BASE_YEARS[-1]}-12-31 22:00", tz="UTC"))
p = smard_prices(pd.Timestamp(f"{BASE_YEARS[0] - 1}-12-31 23:00", tz="UTC"),
                 pd.Timestamp(f"{BASE_YEARS[-1]}-12-31 22:00", tz="UTC"))
print("hours:", len(p), " from", p.index.min(), "to", p.index.max())
print("yearly average EUR/MWh:")
print(p.groupby(p.index.tz_convert(TZ).year).mean().round(2).to_string())

hours: 17520  from 2020-12-31 23:00:00+00:00 to 2022-12-31 22:00:00+00:00
yearly average EUR/MWh:
ms
2021     96.85
2022    235.45


In [4]:
p

ms
2020-12-31 23:00:00+00:00    50.87
2021-01-01 00:00:00+00:00    48.19
2021-01-01 01:00:00+00:00    44.68
2021-01-01 02:00:00+00:00    42.92
2021-01-01 03:00:00+00:00    40.39
                             ...  
2022-12-31 18:00:00+00:00    -1.01
2022-12-31 19:00:00+00:00    -1.39
2022-12-31 20:00:00+00:00    -1.04
2022-12-31 21:00:00+00:00    -1.07
2022-12-31 22:00:00+00:00    -0.68
Name: price, Length: 17520, dtype: float64

In [5]:
HOLIDAYS = [
    # nationwide German public holidays: base period 2021-2022 and delivery year 2024
    "2021-01-01", "2021-04-02", "2021-04-05", "2021-05-01", "2021-05-13", "2021-05-24",
    "2021-10-03", "2021-12-25", "2021-12-26",
    "2022-01-01", "2022-04-15", "2022-04-18", "2022-05-01", "2022-05-26", "2022-06-06",
    "2022-10-03", "2022-12-25", "2022-12-26",
    "2024-01-01", "2024-03-29", "2024-04-01", "2024-05-01", "2024-05-09", "2024-05-20",
    "2024-10-03", "2024-12-25", "2024-12-26",
]


def daytype(local_date, weekday):
    return np.where(np.isin(local_date, HOLIDAYS) | (weekday == 6), "Sun",
            np.where(weekday == 5, "Sat", "Work"))


loc = p.index.tz_convert(TZ)
h = pd.DataFrame({"year": loc.year, "month": loc.month, "hour": loc.hour,
                  "ym": loc.strftime("%Y-%m"), "price": p.to_numpy(),
                  "dtype": daytype(loc.strftime("%Y-%m-%d"), loc.dayofweek)})
print(h["dtype"].value_counts().to_string())

dtype
Work    12240
Sun      2832
Sat      2448


In [6]:
h["z"] = h["price"] / h.groupby("ym")["price"].transform("mean")
print("every month now averages:", round(h.groupby("ym")["z"].mean().max(), 10))

every month now averages: 1.0


In [7]:
h

,year,month,hour,ym,price,dtype,z
0,2021,1,0,2021-01,50.87,Sun,0.963333
1,2021,1,1,2021-01,48.19,Sun,0.912582
2,2021,1,2,2021-01,44.68,Sun,0.846112
3,2021,1,3,2021-01,42.92,Sun,0.812783
4,2021,1,4,2021-01,40.39,Sun,0.764872
...,...,...,...,...,...,...,...
17515,2022,12,19,2022-12,-1.01,Sat,-0.004014
17516,2022,12,20,2022-12,-1.39,Sat,-0.005524
17517,2022,12,21,2022-12,-1.04,Sat,-0.004133
17518,2022,12,22,2022-12,-1.07,Sat,-0.004253


In [8]:
h["w"] = np.where(h["year"] == BASE_YEARS[-1], 1.6, 1.0)
h["zw"] = h["z"] * h["w"]

g = h.groupby(["month", "hour", "dtype"])[["zw", "w"]].sum()
lut = g["zw"] / g["w"]
print("cells:", len(lut), " smallest:", round(lut.min(), 3), " largest:", round(lut.max(), 3))

cells: 864  smallest: 0.151  largest: 1.671


In [9]:
g

zw     w
month hour dtype                 
1     0    Sat    10.772523  11.4
           Sun    11.073451  15.6
           Work   42.719707  53.6
      1    Sat    10.076586  11.4
           Sun    10.708091  15.6
...                     ...   ...
12    22   Sun    10.771912  13.0
           Work   52.058602  56.6
      23   Sat     6.853194  11.0
           Sun     8.790766  13.0
           Work   45.613945  56.6

[864 rows x 2 columns]

In [10]:
lut

month  hour  dtype
1      0     Sat      0.944958
             Sun      0.709837
             Work     0.797009
       1     Sat      0.883911
             Sun      0.686416
                        ...   
12     22    Sun      0.828609
             Work     0.919763
       23    Sat      0.623018
             Sun      0.676213
             Work     0.805900
Length: 864, dtype: float64

In [11]:
out = pd.read_csv(f"{path}/shape_factors.csv")
out["dtype"] = daytype(out["timestamp_local"].str[:10].to_numpy(), out["weekday"].to_numpy())

key = pd.MultiIndex.from_arrays([out["month"], out["hour"], out["dtype"]])
s = pd.Series(lut.reindex(key).to_numpy(), index=out.index)

# normalise each delivery month to 1.0, so the file is a pure shape and nothing else
out["new_shape_factor"] = s / s.groupby(out["month"]).transform("mean")

print("missing cells:", int(out["new_shape_factor"].isna().sum()))
print("monthly averages (all 1.0):")
print(out.groupby("month")["new_shape_factor"].mean().round(6).to_string())

missing cells: 0
monthly averages (all 1.0):
month
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
12    1.0


In [12]:
out

,timestamp_utc,timestamp_local,month,hour,weekday,is_peak,historical_shape_factor,dtype,new_shape_factor
0,2023-12-31T23:00:00Z,2024-01-01T00:00:00+01:00,1,0,0,0,0.832554,Sun,0.701846
1,2023-12-31T23:15:00Z,2024-01-01T00:15:00+01:00,1,0,0,0,0.832554,Sun,0.701846
2,2023-12-31T23:30:00Z,2024-01-01T00:30:00+01:00,1,0,0,0,0.832554,Sun,0.701846
3,2023-12-31T23:45:00Z,2024-01-01T00:45:00+01:00,1,0,0,0,0.832554,Sun,0.701846
4,2024-01-01T00:00:00Z,2024-01-01T01:00:00+01:00,1,1,0,0,0.796187,Sun,0.678689
...,...,...,...,...,...,...,...,...,...
35131,2024-12-31T21:45:00Z,2024-12-31T22:45:00+01:00,12,22,1,0,0.958574,Work,0.938787
35132,2024-12-31T22:00:00Z,2024-12-31T23:00:00+01:00,12,23,1,0,0.860461,Work,0.822569
35133,2024-12-31T22:15:00Z,2024-12-31T23:15:00+01:00,12,23,1,0,0.860461,Work,0.822569
35134,2024-12-31T22:30:00Z,2024-12-31T23:30:00+01:00,12,23,1,0,0.860461,Work,0.822569


In [13]:
out["historical_shape_factor"] = out["new_shape_factor"]
out[["timestamp_utc", "timestamp_local", "month", "hour", "weekday", "is_peak",
     "historical_shape_factor"]].to_csv(f"{path}/shape_factors_v2.csv", index=False)
print("written:", f"{path}/shape_factors_v2.csv", len(out), "rows")

written: C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data/shape_factors_v2.csv 35136 rows
